# Problem Set 2: Fixed Effects and Regression Basics

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due Wednesday 10/7 (beginning of class)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

Fixed effects and regression basics, building on the natality data from PS1.

**Before you do anything else:** go to **File → Save a copy in Drive**.

## Part 0: Set-up

This problem set has two halves. In **Part 1** you will *simulate* data to see three core regression facts with your own eyes: what measurement error does to a coefficient, the Frisch–Waugh theorem, and why clustered standard errors matter. In **Part 2** you will use real American Community Survey (ACS) data to estimate fixed effects regressions of fertility on income.

We set a random seed so that everyone's simulation results are identical (and so are the answer key's).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

np.random.seed(333)   # do not change — makes everyone's random draws identical
print("Ready!")

## Part 1, Exercise 1: Measurement error

We create 1,000 observations where the truth is $y = x + \text{noise}$, so the true coefficient on $x$ is **1**. Then we create noisy ("mis-measured") versions of both variables: `y_me` adds noise to $y$, and `x_me` adds noise to $x$. We regress: (1) `y` on `x`, (2) `y_me` on `x`, and (3) `y` on `x_me`.

In [ ]:
n = 1000
sim = pd.DataFrame({"x": np.random.uniform(size=n)})
sim["y"]    = sim.x + np.random.uniform(size=n) - 0.5
sim["y_me"] = sim.y + np.random.uniform(size=n) - 0.5
sim["x_me"] = sim.x + np.random.uniform(size=n) - 0.5

me1 = smf.ols("y ~ x", sim).fit(cov_type="HC1")
me2 = smf.ols("y_me ~ x", sim).fit(cov_type="HC1")
me3 = smf.ols("y ~ x_me", sim).fit(cov_type="HC1")

print(summary_col([me1, me2, me3], stars=True,
                  model_names=["(1) y~x", "(2) y_me~x", "(3) y~x_me"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (yv, xv, title) in zip(axes, [("y", "x", "y vs x"),
                                      ("y_me", "x", "y_me vs x"),
                                      ("y", "x_me", "y vs x_me")]):
    ax.scatter(sim[xv], sim[yv], s=8, alpha=0.5)
    b = np.polyfit(sim[xv], sim[yv], 1)
    xs = np.linspace(sim[xv].min(), sim[xv].max(), 100)
    ax.plot(xs, np.polyval(b, xs), linewidth=3)
    ax.set_xlabel(xv); ax.set_title(title)
    ax.set_ylim(-1, 2); ax.set_xlim(-0.5, 1.5)
axes[0].set_ylabel("y")
plt.show()

**Question 1.** The true coefficient is 1 in all three cases. Which regression recovers it, which does not, and why? What is the general lesson about measurement error in the *outcome* versus measurement error in the *explanatory variable*?

**Answer:** *[your answer here]*

## Part 1, Exercise 2: The Frisch–Waugh theorem

The truth is now $y = x_1 + 2 x_2 + \text{noise}$, where $x_1$ and $x_2$ are correlated (both are built from the same $z$). We compare: (1) the multivariate regression, (2) the *short* regression of `y` on `x1` alone, and (3) the Frisch–Waugh construction — regress `y` on `x2` and keep the residuals, regress `x1` on `x2` and keep the residuals, then regress residuals on residuals.

In [ ]:
sim2 = pd.DataFrame({"z": np.random.uniform(size=n) * 10})
sim2["x1"] = sim2.z + np.random.uniform(size=n)
sim2["x2"] = sim2.z + np.random.uniform(size=n)
sim2["y"]  = sim2.x1 + 2 * sim2.x2 + np.random.uniform(size=n)

fw1 = smf.ols("y ~ x1 + x2", sim2).fit(cov_type="HC1")

sim2["e_y"]  = smf.ols("y ~ x2",  sim2).fit().resid
sim2["e_x1"] = smf.ols("x1 ~ x2", sim2).fit().resid

fw2 = smf.ols("y ~ x1", sim2).fit(cov_type="HC1")
fw3 = smf.ols("e_y ~ e_x1", sim2).fit(cov_type="HC1")

print(summary_col([fw1, fw2, fw3], stars=True,
                  model_names=["(1) long", "(2) short", "(3) resid-on-resid"]))

**Question 2.** Compare the coefficient on `x1` in column (1), on `x1` in column (2), and on `e_x1` in column (3). Which two match (this is the Frisch–Waugh theorem)? Why is the short regression in column (2) so different — what is this bias called?

**Answer:** *[your answer here]*

## Part 1, Exercise 3: Clustered standard errors

We create 100 observations, then *duplicate every row* — first ×2, then ×100. Duplicating rows adds **zero new information**, but watch what happens to the standard errors if we ignore that. `cov_type="cluster"` is the equivalent of Stata's `cluster()` option.

In [ ]:
sim3 = pd.DataFrame({"x": np.random.uniform(size=100)})
sim3["y"] = sim3.x + np.random.uniform(size=100) * 10
sim3["pers_id"] = range(100)

cl1 = smf.ols("y ~ x", sim3).fit()

sim3_x2 = pd.concat([sim3] * 2, ignore_index=True)
cl2 = smf.ols("y ~ x", sim3_x2).fit()
cl3 = smf.ols("y ~ x", sim3_x2).fit(cov_type="cluster", cov_kwds={"groups": sim3_x2.pers_id})

sim3_x200 = pd.concat([sim3] * 200, ignore_index=True)
cl4 = smf.ols("y ~ x", sim3_x200).fit()
cl5 = smf.ols("y ~ x", sim3_x200).fit(cov_type="cluster", cov_kwds={"groups": sim3_x200.pers_id})

print(summary_col([cl1, cl2, cl3, cl4, cl5], stars=True,
                  model_names=["(1) n=100", "(2) x2", "(3) x2 clust", "(4) x200", "(5) x200 clust"],
                  info_dict={"N": lambda r: f"{int(r.nobs):,}"}))

**Question 3.** Compare the standard errors across the five columns. What happens to the *unclustered* standard errors as we duplicate the data, and why is that misleading? What do the clustered standard errors do instead? When would you cluster in a real analysis?

**Answer:** *[your answer here]*

## Part 2: Fixed effects with real data — fertility and income in the ACS

We now use real American Community Survey data (years 2006–2010; female respondents). The data loads directly from a link — it is ~66 MB, so the next cell can take a minute.

| Variable | Meaning |
|---|---|
| `year` | survey year |
| `statefip` | state (FIPS code) |
| `age` | age |
| `fertyr` | gave birth in the past 12 months (1 = no, 2 = yes) |
| `ftotinc` | total family income (we convert it to units of $10,000) |

We clean the data as follows: keep valid fertility responses, keep ages 18–35, create a `childborn` dummy, and set invalid incomes (≤ 0 or the 9999999 "not reported" code) to missing. Finally we measure income in units of $10,000, so the regression coefficients below are not tiny numbers.

In [ ]:
acs = pd.read_stata("https://raw.githubusercontent.com/thetruejacob/e3h-2026-notebooks/main/data/PS5_ACS_2014_16_18.dta", convert_categoricals=False)
print("Rows:", len(acs))

acs = acs[acs.fertyr.isin([1, 2])].copy()
acs = acs[acs.age.between(18, 35)]
acs["childborn"] = (acs.fertyr == 2).astype(int)
acs.loc[(acs.ftotinc <= 0) | (acs.ftotinc == 9999999), "ftotinc"] = np.nan
acs["ftotinc"] = acs.ftotinc / 10000   # income in $10,000s

print("Rows after cleaning:", len(acs))
acs.describe()

### State-by-year averages

Now we compute, for each state and year, the share of women who gave birth and the average family income (the equivalent of Stata's `collapse`), and plot them — first unweighted, then weighting each state-year dot by its number of observations.

In [ ]:
cell = (acs.groupby(["statefip", "year"])
           .agg(childborn=("childborn", "mean"),
                ftotinc=("ftotinc", "mean"),
                n=("childborn", "size"))
           .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

# unweighted
axes[0].scatter(cell.ftotinc, cell.childborn, s=18)
b = np.polyfit(cell.ftotinc, cell.childborn, 1)
xs = np.linspace(cell.ftotinc.min(), cell.ftotinc.max(), 100)
axes[0].plot(xs, np.polyval(b, xs))
axes[0].set_title("Unweighted")

# weighted by cell size
axes[1].scatter(cell.ftotinc, cell.childborn, s=cell.n / cell.n.max() * 400,
                facecolors="none", edgecolors="C0")
bw = np.polyfit(cell.ftotinc, cell.childborn, 1, w=cell.n)
axes[1].plot(xs, np.polyval(bw, xs))
axes[1].set_title("Weighted by cell size")

for ax in axes:
    ax.set_xlabel("Average family income ($10,000s)")
axes[0].set_ylabel("Share who gave birth")
plt.show()

**Question 4.** Compare the two plots. Why do the fitted lines differ, and which one would you trust more to describe the relationship for the average woman (rather than the average state)?

**Answer:** *[your answer here]*

### Three regressions

Using the *individual-level* data, we run: (1) simple OLS, (2) OLS with state dummies, which is the fixed effects specification, and (3) fixed effects "by hand": demean both variables within state, then regress the demeaned variables. All standard errors are clustered by state. To keep the comparison table readable we hide the individual state dummies from column (2).

In [ ]:
reg_df = acs.dropna(subset=["ftotinc"]).copy()

fe1 = smf.ols("childborn ~ ftotinc", reg_df).fit(
    cov_type="cluster", cov_kwds={"groups": reg_df.statefip})
fe2 = smf.ols("childborn ~ ftotinc + C(statefip)", reg_df).fit(
    cov_type="cluster", cov_kwds={"groups": reg_df.statefip})

# "by hand": demean within state
reg_df["childborn_dm"] = reg_df.childborn - reg_df.groupby("statefip").childborn.transform("mean")
reg_df["ftotinc_dm"]   = reg_df.ftotinc   - reg_df.groupby("statefip").ftotinc.transform("mean")
fe3 = smf.ols("childborn_dm ~ ftotinc_dm", reg_df).fit(
    cov_type="cluster", cov_kwds={"groups": reg_df.statefip})

print(summary_col([fe1, fe2, fe3], stars=True, float_format="%.6f",
                  model_names=["(1) OLS", "(2) state dummies", "(3) demeaned"],
                  regressor_order=["ftotinc", "ftotinc_dm", "Intercept"], drop_omitted=True,
                  info_dict={"State FE": lambda r: "Yes" if ("C(statefip)" in " ".join(r.params.index) or "dm" in " ".join(r.params.index)) else "No",
                             "N": lambda r: f"{int(r.nobs):,}"}))

In [ ]:
# A raw scatter of the demeaned variables is not informative here: childborn is
# binary, so after demeaning every point sits in one of two horizontal bands.
# Instead we use a BINNED scatter ("binscatter"), the standard tool in applied
# economics: split demeaned income into 50 equal-sized bins and plot the mean
# of demeaned childborn within each bin, with the regression line on top.

reg_df["income_bin"] = pd.qcut(reg_df.ftotinc_dm, 50, labels=False, duplicates="drop")
binned = reg_df.groupby("income_bin").agg(x=("ftotinc_dm", "mean"),
                                          y=("childborn_dm", "mean"))

plt.figure(figsize=(7, 4.8))
plt.scatter(binned.x, binned.y, s=25)
xs = np.linspace(binned.x.min(), binned.x.max(), 100)
plt.plot(xs, fe3.params["Intercept"] + fe3.params["ftotinc_dm"] * xs, color="C1",
         label="regression line (from the microdata)")
plt.xlabel("Family income in $10,000s (demeaned within state)")
plt.ylabel("Share who gave birth (demeaned within state)")
plt.legend()
plt.show()

**Question 5a.** Before making the plot above, we tried a *raw* scatter of the demeaned individual-level data — and it just showed two horizontal bands of points (one near 0.9, one near −0.1) with no visible pattern. Explain why a raw scatter of a demeaned *binary* variable must look like that, and why binning fixes it.

**Answer:** *[your answer here]*

**Question 5b.** Compare the income coefficient in columns (1), (2), and (3). Why are (2) and (3) (essentially) identical — what does that tell you about what fixed effects *are*? And what variation is the fixed effects coefficient using that the simple OLS coefficient is not?

**Answer:** *[your answer here]*

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS02"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")